In [ ]:

from numpy import *
set_printoptions(legacy = '1.25')

def f(x): return sin(x)
def g(r): return 1/(1+ exp(-r))
def h(s): return s**2

functions = array([f,g,h])

def df(x): return cos(x)
def dg(r): return g(r)*(1-g(r))
def dh(s): return 2*s

derivatives = array([df,dg,dh])


In [ ]:

# first version: chains

def forward_prop(x_in, functions):
	x = array([x_in])
	for f in functions:
		x_out = f(x_in)
		x = append(x, x_out)
		x_in = x_out
	return x

x_in = pi/4
x = forward_prop(x_in, functions)

print(x)


In [ ]:

# dy/dy = 1
delta_out = 1.0


In [ ]:

# first version: chains

def backward_prop(delta_out, x, derivatives):
	delta = array([delta_out])
	# discard last element then reverse x
	# also reverse derivatives
	for a, df in  zip(flip(x[:-1]), flip(derivatives)):
		# chain rule -- multiply by previous der
		der = df(a) * delta[0]
		delta = insert(delta, 0, der) # insert at start
	return delta
	
delta = backward_prop(delta_out, x, derivatives) 
print(delta)


In [ ]:

d = 3
functions, derivatives = array([h]*d), array(dh]*d)
x_in, delta_out = 5, 1

x = forward_prop(x_in, functions)
delta = backward_prop(delta_out, x, derivatives) 

print(x, delta)


In [ ]:

d = 7
w = full((d,d), None)
# array indexing ranges from 0 to 6, not 1 to 7

w[3,0] = w[3,1] = w[4,1] = w[4,2] = 1
w[5,3] = w[5,4] = w[6,5] = 1

print(w)


In [ ]:

activate = full(d, None)
# array indexing ranges from 0 to 6, not 1 to 7

activate[3] = lambda x,y: x+y
activate[4] = lambda y,z: max(y,z)
activate[5] = lambda a,b: a*b
print(activate)


In [ ]:

def incoming(x, w, i):
	return [ w[i,j] * outgoing(x, w, j) for j in range(d) if w[i,j] ]


In [ ]:

def outgoing(x, w, i):
	if x[i] != None: return x[i]
	elif activate[i]: return activate[i](*incoming(x, w, i))
	else: return None


In [ ]:

# second version: networks

def forward_prop(x_in, w):
	d = len(w)
	x = full(d, None)
	m = len(x_in)
	x[:m] = x_in
	for i in range(m,d): x[i] = outgoing(x, w, i)
	return x
	
x_in = array([1, 2, 0])
x = forward_prop(x_in, w)

print(x)


In [ ]:

g = full((d,d), None)
# array indexing ranges from 0 to 6, not 1 to 7

g[3,0] = lambda x,y: 1
g[3,1] = lambda x,y: 1
g[4,1] = lambda y,z: 1 if y >= z else 0
g[4,2] = lambda y,z: 1 if z > y else 0
g[5,3] = lambda a,b: b
g[5,4] = lambda a,b: a

print(g)


In [ ]:

def derivative(x, delta, g, j):
	if delta[j] != None: return delta[j]
	else:
		return sum([ derivative(x, delta, g, i) * g[i,j]( *incoming(x, w, i)) * w[i,j] for i in range(d) if g[i,j] ] )


In [ ]:

# second version: networks

def backward_prop(x, delta_out, g):
	d = len(g)
	delta = full(d, None)
	m = len(delta_out)
	delta[-m:] = delta_out
	for j in range(d-m): 
		delta[j] = derivative(x, delta, g, j)
	return delta


In [ ]:

delta_out = array([1,None])
delta = backward_prop(x, delta_out, g)

print(delta)
